# Define what better means

A parser that gets `"30s"` right can still get `"2h"` wrong. Before improving
it, you need a way to measure its answers and decide which revision to keep.
An **evaluator** computes measurements; an **objective** says which measurement
to improve and whether higher or lower is better.
The measurements come from the evaluator, independently of a proposer's claims;
see [evaluation authority](https://sentient-xyz.github.io/meta-evolve-docs/concepts/authority/).

| You want to… | Use |
|---|---|
| [Grade answers](#2-choose-the-answers-that-matter) | An evaluator returning a score |
| [Track several measurements](#4-report-named-measurements) | A dictionary of metrics |
| [Choose what determines the winner](#5-declare-which-metric-determines-better) | `Maximize` or `Minimize` |

The revisions are handwritten. Python executes and grades
them; no model, SDK, or credentials are needed.



<a id="1-install"></a>

## Required setup for a fresh notebook

Use a fresh notebook environment running **Python 3.12 or newer**.
Install directly from the published documentation:

In [ ]:
%pip install https://sentient-xyz.github.io/meta-evolve-docs/downloads/meta-evolve.zip

If you already imported Meta-Evolve, restart the kernel after installing.
Then run the remaining cells in order.

**Archived or offline docs:** use the ZIP included with that build. Put
`meta-evolve.zip` in the notebook's working folder (`%pwd` shows it; hosted
notebooks let you upload files), then run `%pip install ./meta-evolve.zip`
instead. Installing from source may still download build tools.

**Starting source.** `SEED` reads a duration's number but ignores its unit.
`CONTRACT` tells the proposer what the parser should do.

In [ ]:
import meta_evolve as meta

CONTRACT = """Implement parse_seconds(text) for whole-number durations.
Inputs contain a number followed by s, m, or h, such as '30s' or '2h'.
Return the duration in seconds as an integer. Return only Python source.
"""

SEED = '''def parse_seconds(text):
    return int(text[:-1])
'''

**Revisions.** `MINUTES` adds minutes support; `COMPLETE` adds hours as well.

In [ ]:
MINUTES = '''def parse_seconds(text):
    quantity = int(text[:-1])
    return quantity * 60 if text[-1] == "m" else quantity
'''

COMPLETE = '''def parse_seconds(text):
    quantity = int(text[:-1])
    seconds_per_unit = {"s": 1, "m": 60, "h": 3600}
    return quantity * seconds_per_unit[text[-1]]
'''

**Proposer.** `propose` sends the contract and source to a simulated agent.
Given `SEED`, it returns `MINUTES`; given `MINUTES`, it returns `COMPLETE`.
Its choice depends on the source, without a hidden response counter.

In [ ]:
def agent(message):
    """Simulate source revisions, without a model or a call counter."""
    source = message.rsplit("Current source:\n", 1)[1]
    if source == SEED:
        return MINUTES
    if source in (MINUTES, COMPLETE):
        return COMPLETE
    raise ValueError("This simulation only recognizes the tutorial sources.")


def propose(source):
    message = f"{CONTRACT}\nCurrent source:\n{source}"
    return agent(message)

The [Start here walkthrough](https://sentient-xyz.github.io/meta-evolve-docs/start-here/) introduces the starting parser.


<a id="2-choose-the-answers-that-matter"></a>

## 1. Choose the answers that matter

An evaluator can be an ordinary Python function. Ours loads the parser, checks
six expected answers, and returns the fraction correct. A score of `1.0` means
all six checks pass. `load_parser` executes the source to obtain the function;
this small evaluator is for the reviewed, handwritten code shown here.

In [ ]:
CASES = (
    ("30s", 30),
    ("90s", 90),
    ("2m", 120),
    ("3m", 180),
    ("1h", 3600),
    ("2h", 7200),
)


def load_parser(source):
    namespace = {}
    exec(source, namespace)
    return namespace["parse_seconds"]


def evaluate(source):
    parse = load_parser(source)
    passed = 0
    for text, expected in CASES:
        actual = parse(text)
        if type(actual) is int and actual == expected:
            passed += 1
    return passed / len(CASES)


print(f"Starting score: {evaluate(SEED):.0%}")
# Output:
# Starting score: 33%

Only the two seconds checks pass. These expected answers belong to the evaluator:
a proposed parser revision cannot change them. Choose cases that represent the
behavior you need; this six-case demonstration measures only the cases shown.

<a id="3-propose-revisions-and-select-the-best-score"></a>

## 2. Select the best score

[`meta.improve()`][meta_evolve.improve] evaluates the seed, proposes revisions
from the best version so far, and evaluates each valid revision. With a scalar
evaluator, the measurement is named `score` and larger values win by default.
`trials=2` allows two proposal attempts after the seed evaluation.

In [ ]:
result = meta.improve(
    seed=SEED,
    proposer=propose,
    evaluator=evaluate,
    trials=2,
)

The returned run keeps each attempt and its measurements. `trials()` lists
the seed followed by the revisions; `best_trial()` gives the selected attempt.

In [ ]:
for number, attempt in enumerate(result.trials()):
    print(f"Version {number}: {attempt.metrics['score']:.0%}")
print(f"Selected score: {result.best_trial().metrics['score']:.0%}")
# Output:
# Version 0: 33%
# Version 1: 67%
# Version 2: 100%
# Selected score: 100%

The proposer supplies source, and the evaluator independently measures it. The
100% score comes from running the checks on `COMPLETE`.

<a id="4-report-named-measurements"></a>

## 3. Report named measurements

An overall score can hide a weak category. Return a dictionary to record named
metrics from the same six checks. We will retain overall `score`, accuracy on
`hours`, and the number of `wrong_answers`.

In [ ]:
def evaluate_metrics(source):
    parse = load_parser(source)
    passed = 0
    hours_passed = 0
    hours_count = 0
    for text, expected in CASES:
        actual = parse(text)
        correct = type(actual) is int and actual == expected
        passed += correct
        if text.endswith("h"):
            hours_count += 1
            hours_passed += correct
    return {
        "score": passed / len(CASES),
        "hours": hours_passed / hours_count,
        "wrong_answers": len(CASES) - passed,
    }

for source in (SEED, MINUTES, COMPLETE):
    metrics = evaluate_metrics(source)
    print(f"Overall: {metrics['score']:.0%}; hours: {metrics['hours']:.0%}; "
          f"wrong answers: {metrics['wrong_answers']}")
# Output:
# Overall: 33%; hours: 0%; wrong answers: 4
# Overall: 67%; hours: 0%; wrong answers: 2
# Overall: 100%; hours: 100%; wrong answers: 0

The minutes revision improves the overall score while still failing both hours
checks. Recording `hours` makes that gap visible. It does not automatically make
hours accuracy a selection objective.

<a id="5-declare-which-metric-determines-better"></a>

## 4. Declare which metric determines better

Pass one objective to [`meta.improve()`][meta_evolve.improve].
[`meta.Maximize`][meta_evolve.Maximize] selects higher values of a named metric;
[`meta.Minimize`][meta_evolve.Minimize] selects lower values.

Use one proposal attempt here so the hours gap remains visible. **Greedy**
search builds on the best measured version, just as `improve()` does by default.

In [ ]:
metrics_result = meta.improve(
    seed=SEED, proposer=propose, evaluator=evaluate_metrics, trials=1,
    objective=meta.Maximize("score"),
)
selected = metrics_result.best_trial().metrics
print(f"Overall: {selected['score']:.0%}; hours: {selected['hours']:.0%}")
# Output:
# Overall: 67%; hours: 0%

The selected parser is `MINUTES`, because `score` is the objective. Every metric
remains available for inspection, including the hours result that did not improve.
If your evaluator uses a name other than `score`, declare the matching objective.
A scalar evaluator measures the declared objective directly; a mapping must
include its exact name. See the [scalar minimization example](https://sentient-xyz.github.io/meta-evolve-docs/reference/api-examples/#one-objective-on-the-short-path).

For reusable declarations or several ordered objectives, use
[`meta.Task`][meta_evolve.Task] and [`meta.Experiment`][meta_evolve.Experiment].
This expanded declaration executes the same search:

In [ ]:
metrics_experiment = meta.Experiment(
    seed=SEED,
    proposer=propose,
    task=meta.Task(
        evaluator=evaluate_metrics,
        objectives=(meta.Maximize("score"),),
    ),
    search=meta.Greedy(max_trials=1),
)

expanded_metrics = meta.run(metrics_experiment)

For a measurement where less is better, change the objective to `Minimize`:

In [ ]:
errors_result = meta.improve(
    seed=SEED, proposer=propose, evaluator=evaluate_metrics, trials=1,
    objective=meta.Minimize("wrong_answers"),
)
print("Selected wrong answers:", errors_result.best_trial().metrics["wrong_answers"])
# Output:
# Selected wrong answers: 2

With these fixed six cases, fewer wrong answers and a higher overall score
select the same revision. Other measurements, such as runtime, may reward
different candidates; choose an objective that matches your actual goal.

An exception while executing or grading source produces a failed evaluation
with no score. A measured wrong answer contributes to the score above.
[Inspect results and failures](https://sentient-xyz.github.io/meta-evolve-docs/learn/03-inspect-results/) explains how to
tell those outcomes apart.

## Change and predict

In the [objective declaration](#5-declare-which-metric-determines-better), change
`meta.Maximize("score")` to `meta.Maximize("hours")`, keeping the one-attempt
limit. Predict the selected score, then rerun that declaration and its result cell.

The printed result becomes **Overall: 33%; hours: 0%**. The seed and minutes
revision both score zero on hours, so the revision does not beat the seed on
the declared objective. The seed stays selected even though the revision has
a higher overall score. This shows why the objective matters separately from
the list of measurements you record.

## Why didn't my new metric change the winner?

Returning a metric records it. The objective decides whether it influences
selection; adding `hours` to the dictionary alone leaves `score` in charge.
Use the metric's exact name in the objective, and choose its direction.

For your own task, replace `CASES` and the evaluator with independent checks
that represent your goal. Replace the source and proposer with your starting
value and revision function. Keep grading outside the proposer, and try the
evaluator on a known good and a known bad value before starting a search.

Next, [give the proposer feedback](https://sentient-xyz.github.io/meta-evolve-docs/learn/05-use-experience/): record which
checks failed so the next revision can address them. See the
[evaluation reference](https://sentient-xyz.github.io/meta-evolve-docs/guides/evaluation-and-budgets/) for additional result forms
and ordered objectives. To understand the observations behind a score, read
[Evidence](https://sentient-xyz.github.io/meta-evolve-docs/concepts/evidence/).